# Import packages and load data

In [ ]:
import pandas as pd
import numpy as np
import time
import sklearn
from sklearn.ensemble import RandomForestRegressor
from skopt import BayesSearchCV
from skopt.space import Integer, Real, Categorical
from sklearn.model_selection import train_test_split
from sklearn import model_selection
from sklearn.preprocessing import StandardScaler
from joblib import dump, load

# Data pre-processing

In [ ]:
output = 'gisaxs_domain'
data_directory = '../../chemical-featurization/full_descriptors/poly_maccs.xlsx'
df_likeness = pd.read_excel(data_directory)

# Bayesian-Search

In [ ]:
def bayes_train(df, output):

    df_clean = df.dropna()

    inputs = df_clean.iloc[:, 1:-2].copy()
    target = df_clean[f"{output}"]

    X_train, X_test, y_train, y_test = train_test_split(
        inputs, target, test_size=0.2, random_state=10
    )
    
    # Scale the inputs (fit only on training data to avoid data leakage)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
    X_test_scaled = scaler.transform(X_test)
    X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

    RANDOM_SEED = 42
    CV_FOLDS = 5

    search_spaces = {

        # forest size (log scaling explores wide range efficiently)
        "n_estimators": Integer(50, 300),

        # tree complexity
        "max_depth": Integer(3, 20),

        # regularization knobs
        "min_samples_split": Integer(2, 20),
        "min_samples_leaf": Integer(1, 12),

        # feature sampling
        "max_features": Categorical(["sqrt", "log2", 1.0, 0.3, 0.5, 0.8]),

        # bootstrap behavior
        "bootstrap": Categorical([True, False]),

        # split criterion
        "criterion": Categorical(["squared_error", "friedman_mse"]),
    }

    rf = RandomForestRegressor(
        random_state=RANDOM_SEED,
        n_jobs=-1
    )

    opt = BayesSearchCV(
        estimator=rf,
        search_spaces=search_spaces,
        n_iter=60,                 # number of Bayesian steps
        scoring="r2",
        cv=CV_FOLDS,
        n_jobs=-1,
        random_state=RANDOM_SEED,
        verbose=0
    )

    # Fit optimizer
    opt.fit(X_train_scaled, y_train)

    best_model = opt.best_estimator_
    best_cv_r2 = opt.best_score_
    best_test_r2 = best_model.score(X_test_scaled, y_test)

    print("CV Train R²:", best_cv_r2)
    print("Test R²:", best_test_r2)
    print("Best Parameters:", opt.best_params_)

    return best_model

In [ ]:
%%time

best_model = bayes_train(df_likeness, output)

In [ ]:
df_likeness_key = "poly_maccs"
dump(best_model , f'./full_descriptors/{output}_{df_likeness_key}.joblib')